# Benchmark de Cascata Adaptativa — Marco 3 (pivot NCM)

Validação no Google Colab (GPU NVIDIA T4) do pivot de dataset
: classificação de itens de catálogo em
códigos NCM (a taxonomia fiscal brasileira em árvore) — o tipo de conteúdo
ambíguo de *long tail* que o blog do iFood (Sugar/Revisor/Pepper) descreve.

- **B** — servidor vLLM com prefix caching (inalterado do Marco 1)
- **Sonda** — estatística do pool NCM + granularidade do rótulo por medição
- **C** — System 1: geração curta do código NCM com confiança por logprobs
- **F** — System 2: CoT + JSON com código NCM
- **G** — benchmark: 3 cenários, sweep de α, gates SC-A/B/C e artefatos

**Como usar**: em *Runtime → Change runtime type*, selecione **T4 GPU** e
rode as células em ordem. A sonda decide a granularidade (`LABEL_DIGITS`)
usada por todas as células seguintes.

Nota de reprodutibilidade: a documentação do vLLM avisa que resultados
podem variar entre requests idênticas (batching/prefix cache).
`VLLM_BATCH_INVARIANT=1` **não** é configurado porque o benchmark mede
latência realista.

## Setup — clone e dependências

Clona o repo público e instala `requirements-colab.txt` (inclui o vLLM; a instalação demora alguns minutos).

In [ ]:
import os
import subprocess

REPO = "adaptive-cascade-routing"
BRANCH = "005-dataset-pivot-ncm"  # PR do pivot NCM; volta para "main" após o merge
REPO_URL = "https://github.com/Grazinascito/adaptive-cascade-routing.git"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO, check=True)

In [ ]:
%cd adaptive-cascade-routing
!git log --oneline -1
!pip install -q -r requirements-colab.txt

In [ ]:
import os
import sys

# Guard contra execução fora de ordem (ex.: runtime reiniciado sem
# rodar o Setup): garante que o cwd é o repo clonado antes de
# registrar src/ no sys.path.
if os.path.exists("adaptive-cascade-routing"):
    os.chdir("adaptive-cascade-routing")
sys.path.insert(0, os.getcwd())

from src.data.loader import (
    chapter_lines,
    get_frozen_eval_set,
    is_valid_code,
    load_ncm_pool,
    pool_stats,
    truncate_code,
)
from src.engine import server, system1, system2
from src.metrics.evaluator import (
    JSON_GATE_TARGET,
    BenchmarkHarness,
    confidence_degeneration_share,
    plot_tradeoff_curve,
    save_raw_results,
    sweep_alpha,
)
from src.metrics.summary import compute_summary_metrics, evaluate_kpis

## Componente B — servidor vLLM (Story 1)

Sobe o servidor com `Qwen/Qwen3-4B-AWQ` (AWQ 4-bit, arquitetura `Qwen3ForCausalLM` — 4 bits para caber na VRAM da T4 de 16 GB), prefix caching e 90% da VRAM (limite de VRAM: ≤ 14.4 GB da T4 de 16 GB). Na primeira execução o vLLM baixa ~2.7 GB de pesos, por isso o timeout de startup aqui é 1800 s. Se algo falhar, o log completo do servidor fica em `vllm_server.log`.


In [ ]:
started = server.start_server(system1.DEFAULT_MODEL, startup_timeout_s=1800.0)
print("start_server:", started)
print("health_check:", server.health_check())
print("log do servidor:", server.VLLM_LOG_PATH)

In [ ]:
warmup_item = "ALCOOL LIQUIDO START NAT 70% - 5VL"
latencies = []
for i in range(10):
    output = system1.evaluate_short_classification(warmup_item)
    latencies.append(output["latency_ms"])
    print(f"warmup {i + 1:>2}: {output['latency_ms']:.1f} ms  code={output['code']}")

rest_avg = sum(latencies[1:]) / len(latencies[1:])
drop = 100.0 * (latencies[0] - rest_avg) / latencies[0]
print(f"\nqueda média das queries 2-10 vs a 1ª: {drop:.1f}% (Independent Test: >= 70%)")

In [ ]:
!nvidia-smi
print("limite de VRAM: 14.4 GB (90% da T4 de 16 GB) para pesos + KV cache")

## Componente C — System 1 NCM: sonda + geração curta

A sonda (Stories 1 e 3 da spec do pivot) carrega o pool, imprime a
estatística de extração e decide a granularidade do rótulo: o nível mais
fino (2, 4 ou 8 dígitos) em que a acurácia do System 1 fica na faixa
0.30–0.70 — o terreno com ambiguidade real para este par modelo+tarefa.
O gate anti-degeneração avisa se a confiança do S1 estiver >= 0.95 em
quase tudo (o cenário que o Marco 1 mediu no ASSIN 2, sem faixa de
ambiguidade).

In [ ]:
# Sonda: pool (primeira execução baixa ~27 MB e parseia ~105k linhas) +
# granularidade por medição. A mesma passada alimenta o gate de extração.
POOL, N_ROWS = load_ncm_pool()
STATS = pool_stats(POOL, n_rows=N_ROWS)
print("== SONDA 1: estatística do pool ==")
print(
    f"  linhas={STATS['n_rows']} extraidas={STATS['n_parsed']}"
    f" taxa={STATS['extraction_rate']:.4f} (gate >= 0.98)"
)
print(
    f"  pares únicos={STATS['unique_pairs']} códigos={STATS['unique_codes']}"
    f" capítulos={len(chapter_lines())}"
)
print(
    f"  nomes conflitantes={STATS['conflicting_names']} nomes curtos={STATS['dropped_short_names']}"
)
assert STATS["extraction_rate"] >= 0.98, STATS["extraction_rate"]

probe_samples = get_frozen_eval_set(POOL, sample_limit=100, head_share=0.65, seed=42)
probe_harness = BenchmarkHarness(probe_samples)
try:
    probe_run = probe_harness.run_full_benchmark(confidence_threshold=0.85)
finally:
    probe_harness.close()
probe_s1 = probe_run["records"]["system1_pure"]

print("\n== SONDA 2: granularidade (acurácia S1 por nível) ==")
LABEL_DIGITS = 8
for digits in (2, 4, 8):
    hits = sum(
        truncate_code(r["prediction"], digits) == truncate_code(r["ground_truth"], digits)
        for r in probe_s1
    )
    acc = hits / len(probe_s1)
    in_range = 0.30 <= acc <= 0.70
    marker = " <- faixa 0.30-0.70" if in_range else ""
    print(f"  {digits} dígitos: acc={acc:.2f}{marker}")
    if in_range:
        LABEL_DIGITS = digits  # fica com o MAIS FINO que cai na faixa
print(f"\nLABEL_DIGITS = {LABEL_DIGITS} (o mais fino com S1 em 0.30-0.70)")

degen = confidence_degeneration_share(probe_run)
print(f"gate anti-degeneração: {degen:.2f} dos itens com confiança S1 >= 0.95")
if degen > 0.95:
    print("AVISO: sem faixa de ambiguidade — parar antes do run completo (spec)")

## Componente F — System 2 NCM: CoT + JSON (Story 5)

O System 2 recebe o nome do item com a lista de capítulos como contexto da
taxonomia (o "mapa da árvore") e devolve JSON `{"reasoning", "ncm_code"}`.
Gate: >= 95% de JSON válido sem fallback de regex, e o código precisa
existir na taxonomia.

In [ ]:
eval20 = get_frozen_eval_set(POOL, sample_limit=20, head_share=0.65, seed=42)
outputs = [system2.execute_reasoning(s["name"]) for s in eval20]

valid = sum(not o["fallback_used"] for o in outputs)
codes_ok = sum(is_valid_code(o["ncm_code"]) for o in outputs)
print(f"JSON válido sem regex: {valid}/{len(outputs)} = {valid / len(outputs):.1%} (gate >= 95%)")
print(f"código existente na taxonomia: {codes_ok}/{len(outputs)}")
lat = [o["latency_ms"] for o in outputs]
print(f"latência System 2 — min {min(lat):.0f} ms / max {max(lat):.0f} ms")
for o in outputs[:2]:
    print("-", o["ncm_code"], "| fallback:", o["fallback_used"], "|", o["reasoning"][:80])

## Componente G — benchmark completo (Story 6)

800 itens congelados (~65% cabeça / ~35% cauda por frequência do código no
pool), 3 cenários, sweep de α por replay e os gates do pivot:

- **SC-A**: gap de acurácia S2−S1 >= 10 pp na cauda
- **SC-B**: cascata recupera >= 50% do gap da cauda sem cair abaixo do S1
  puro no conjunto completo
- **SC-C**: p95 e decode tokens >= 50% / 60% menores que o System 2 puro

A taxa de fallback no α ótimo é comparada à banda 30–50% do blog do iFood.

In [ ]:
# G1 (dry-run): 12 amostras nos 3 cenários; todo item_id precisa aparecer
# nos 3. LABEL_DIGITS vem da sonda.
from pathlib import Path

dry_samples = get_frozen_eval_set(POOL, sample_limit=12, head_share=0.65, seed=42)
dry_harness = BenchmarkHarness(dry_samples, label_digits=LABEL_DIGITS)
try:
    dry_run = dry_harness.run_full_benchmark(confidence_threshold=0.85)
finally:
    dry_harness.close()

ids = {s["item_id"] for s in dry_samples}
missing = {
    scenario: ids - {r["query_id"] for r in records}
    for scenario, records in dry_run["records"].items()
}
save_raw_results(
    Path("results/raw_benchmark_data_dryrun.json"),
    meta={
        "model": system1.DEFAULT_MODEL,
        "frozen_samples": len(dry_samples),
        "label_digits": LABEL_DIGITS,
        "dry_run": True,
    },
    scenarios=dry_run["records"],
    alpha_sweep={},
    summaries={},
    kpis=None,
)

print("== G1: dry-run (12 amostras) ==")
for scenario, records in dry_run["records"].items():
    print(f"  {scenario}: {len(records)} registros, ausentes={sorted(missing[scenario])}")
assert all(not miss for miss in missing.values()), missing
print("OK — todo item_id presente nos 3 cenários")

In [ ]:
# G2: benchmark completo — 800 amostras congeladas (65/35), α do config.
import os

os.environ["HF_DATASETS_OFFLINE"] = "1"  # pool já em cache da sonda

KPIS = {
    "p95_reduction": 0.50,
    "token_savings": 0.60,
    "tail_gap_pp": 10,
    "recovery_share": 0.50,
}
full_samples = get_frozen_eval_set(POOL, sample_limit=800, head_share=0.65, seed=42)
full_harness = BenchmarkHarness(full_samples, label_digits=LABEL_DIGITS)
try:
    full_run = full_harness.run_full_benchmark(confidence_threshold=0.85)
finally:
    full_harness.close()

s1_summary = compute_summary_metrics(full_run["records"]["system1_pure"], warmup_discard=5)
s2_summary = compute_summary_metrics(full_run["records"]["system2_pure"], warmup_discard=5)
cascade_summary = compute_summary_metrics(
    full_run["records"]["cascade"],
    warmup_discard=5,
    system1_accuracy=s1_summary["accuracy"],
    system2_accuracy=s2_summary["accuracy"],
)
kpi_report = evaluate_kpis(cascade_summary, s1_summary, s2_summary, KPIS)


def _acc(value):
    return "n/a" if value is None else f"{value:.3f}"


print("== G2: benchmark completo (800 amostras, α=0.85) ==")
for name, s in (
    ("system1_pure", s1_summary),
    ("system2_pure", s2_summary),
    ("cascade", cascade_summary),
):
    p = s["latency_percentiles_ms"]
    print(
        f"  {name}: n={s['n_requests']} p95={p['p95']:.1f}ms"
        f" acc={s['accuracy']:.3f} (head={_acc(s['accuracy_head'])}"
        f" tail={_acc(s['accuracy_tail'])}) f1={s['f1_macro']:.3f}"
        f" decode={s['tokens_decode_total']} taxaS2={s['system2_call_rate']:.2f}"
    )
json_rate = s2_summary["json_success_rate"]
print(f"  gate F (json_success_rate): {json_rate:.3f} vs alvo >= {JSON_GATE_TARGET}")
print("\n== KPIs do pivot ==")
for key, (passed, value) in kpi_report.items():
    print(f"  {key}: {value:.3f} → {'PASS' if passed else 'FAIL'}")

In [ ]:
# G3: sweep de α por replay sobre as medições do benchmark completo.
sweep = sweep_alpha(full_run, label_digits=LABEL_DIGITS)
sweep_summaries = {
    alpha: compute_summary_metrics(
        records,
        warmup_discard=5,
        system1_accuracy=s1_summary["accuracy"],
        system2_accuracy=s2_summary["accuracy"],
    )
    for alpha, records in sweep.items()
}

print("== G3: sweep de α (replay) ==")
rates = []
for alpha in sorted(sweep_summaries):
    s = sweep_summaries[alpha]
    rates.append(s["system2_call_rate"])
    print(
        f"  α={alpha}: taxaS2={s['system2_call_rate']:.2f}"
        f" p95={s['latency_percentiles_ms']['p95']:.1f}ms"
        f" acc={s['accuracy']:.3f} decode={s['tokens_decode_total']}"
    )
monotonic = "OK" if rates == sorted(rates) else "VIOLADA"
print(f"  monotonicidade taxaS2: {monotonic} ({[round(r, 2) for r in rates]})")
qualifying = [
    a for a in sorted(sweep_summaries) if sweep_summaries[a]["accuracy"] >= s1_summary["accuracy"]
]
if qualifying:
    optimal = min(qualifying, key=lambda a: sweep_summaries[a]["system2_call_rate"])
    rate = sweep_summaries[optimal]["system2_call_rate"]
    band = "dentro" if 0.30 <= rate <= 0.50 else "fora"
    print(f"  α ótimo: {optimal} (taxaS2 {rate:.2f} — {band} da banda 30-50% do blog)")
else:
    print("  α ótimo: nenhum α vence o S1 puro")

In [ ]:
# Artefatos (Story 6): log bruto + curva de trade-off.
from datetime import UTC, datetime
from pathlib import Path

save_raw_results(
    Path("results/raw_benchmark_data.json"),
    meta={
        "model": system1.DEFAULT_MODEL,
        "base_url": system1.DEFAULT_BASE_URL,
        "dataset": "yurifacanha/NCM-RAFT-samall",
        "pool_stats": dict(STATS),
        "frozen_samples": len(full_samples),
        "seed": 42,
        "head_share": 0.65,
        "label_digits": LABEL_DIGITS,
        "alpha_default": 0.85,
        "alphas": sorted(sweep_summaries),
        "warmup_discard": 5,
        "degeneration_share": confidence_degeneration_share(full_run),
        "generated_at_utc": datetime.now(UTC).isoformat(),
    },
    scenarios=full_run["records"],
    alpha_sweep={str(alpha): records for alpha, records in sweep.items()},
    summaries={
        "system1_pure": s1_summary,
        "system2_pure": s2_summary,
        "cascade": cascade_summary,
        "alpha_sweep": {str(alpha): s for alpha, s in sweep_summaries.items()},
    },
    kpis=kpi_report,
)
plot_tradeoff_curve(Path("results/latency_cost_tradeoff.png"), full_run, sweep_summaries)

print("== G4: artefatos ==")
for name in ("raw_benchmark_data.json", "latency_cost_tradeoff.png"):
    size = Path("results", name).stat().st_size
    print(f"  results/{name}: {size:,} bytes")

## Próxima fase

Com o benchmark NCM validado e os artefatos commitados, o Marco 4 (ciclo de
agentes de auditoria) roda por cima do `results/raw_benchmark_data.json`:
juiz LLM (branch `006-judge-audit`), loop de insights (`007-insights-loop`)
e HITL (`008-hitl-review`).